# Measuring the Balmer break of MoM-BH\*-1

**Goal:** reproduce the Balmer break strength in the discovery paper (Naidu et al.): **7.7 (+2.3 / −1.4)**.

**What we measure:** the average brightness in a window just *redward* of the Balmer limit (4000–4100 Å, rest frame) divided by the average brightness in a window just *blueward* of it (3620–3720 Å, rest frame). Ordinary star populations can't go above about 3, and pure A-type stars stay below 5.

**How to use this notebook:** click inside each grey code cell, from top to bottom, and press **Shift + Enter** to run it. Wait for each cell to finish before running the next one.

*Data: JWST NIRSpec prism spectra from Naidu et al., Zenodo record 15059215 (CC BY 4.0 — free to reuse with credit).*

## Step 1 — Download the two spectra

This downloads both files from Zenodo into Colab. One is MoM-BH\*-1; the other is most likely its neighbouring galaxy. Colab deletes files when the session ends, so run this cell again each time you come back.

In [ ]:
!wget -q -O spec_150135.fits "https://zenodo.org/records/15059215/files/mom-uds01-v3_prism-clear_5224_150135.spec.fits?download=1"
!wget -q -O spec_948311.fits "https://zenodo.org/records/15059215/files/mom-uds01-v3_prism-clear_5224_948311.spec.fits?download=1"

# Check that both downloads worked (each file should be about 630 KB)
import os
for name in ["spec_150135.fits", "spec_948311.fits"]:
    size_kb = os.path.getsize(name) / 1024 if os.path.exists(name) else 0
    if size_kb > 300:
        print(f"{name}: {size_kb:.0f} KB -> OK")
    else:
        print(f"{name}: {size_kb:.0f} KB -> download failed, run this cell again in a few minutes")

## Step 2 — Set up the measurement

This cell doesn't print anything. It just prepares three small tools:

1. **`load_spectrum`** opens a file and pulls out the wavelength (µm), brightness (µJy) and error bars, plus the object's position in the sky.
2. **`window_average`** turns a rest-frame window into observed wavelengths (multiply by 1 + z) and averages the brightness inside it.
3. **`balmer_break`** divides the red-window average by the blue-window average. For the error bars, it repeats that division 10,000 times with random noise added (matching the error bars) and looks at the spread of answers. This is called a Monte Carlo error, and it naturally gives uneven errors like the paper's +2.3 / −1.4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

Z = 7.7569                  # redshift of MoM-BH*-1 from the paper
BLUE_WINDOW = (3620, 3720)  # rest-frame Å, just blueward of the Balmer limit (3647 Å)
RED_WINDOW = (4000, 4100)   # rest-frame Å, just redward of the break


def load_spectrum(name):
    '''Return wavelength (µm), flux and error (µJy), and the sky position (RA, Dec).'''
    with fits.open(name) as f:
        ext = "SPEC1D" if "SPEC1D" in f else 1      # the 1D spectrum table
        tab = Table(f[ext].data)
        ra = dec = None
        for hdu in f:                               # find the stored sky position
            if "SRCRA" in hdu.header:
                ra, dec = hdu.header["SRCRA"], hdu.header["SRCDEC"]
                break
    cols = {c.lower(): c for c in tab.colnames}     # column names, ignoring upper/lower case
    err_col = cols.get("err", cols.get("full_err"))
    wave = np.array(tab[cols["wave"]], dtype=float)
    flux = np.array(tab[cols["flux"]], dtype=float)
    err = np.array(tab[err_col], dtype=float)
    if np.nanmedian(wave) > 100:                    # if wavelengths are in Å, convert to µm
        wave = wave / 1e4
    good = np.isfinite(wave) & np.isfinite(flux) & np.isfinite(err) & (err > 0)
    return wave[good], flux[good], err[good], ra, dec


def window_average(wave, flux, err, window):
    '''Average flux and its error inside a rest-frame window given in Å.'''
    lo, hi = [w * (1 + Z) / 1e4 for w in window]   # rest-frame Å -> observed µm
    inside = (wave >= lo) & (wave <= hi)
    n = int(inside.sum())
    mean = flux[inside].mean()
    mean_err = np.sqrt(np.sum(err[inside] ** 2)) / n
    return mean, mean_err, n


def balmer_break(wave, flux, err, n_trials=10000, seed=42):
    '''Break strength = red-window average / blue-window average, with Monte Carlo errors.'''
    red, red_err, n_red = window_average(wave, flux, err, RED_WINDOW)
    blue, blue_err, n_blue = window_average(wave, flux, err, BLUE_WINDOW)
    rng = np.random.default_rng(seed)
    trials = rng.normal(red, red_err, n_trials) / rng.normal(blue, blue_err, n_trials)
    low, mid, high = np.percentile(trials, [16, 50, 84])
    return dict(red=red, red_err=red_err, n_red=n_red, blue=blue, blue_err=blue_err,
                n_blue=n_blue, value=mid, plus=high - mid, minus=mid - low)

## Step 3 — Measure both spectra

This measures the break in both files. **MoM-BH\*-1 is the one at RA ≈ 34.3975, Dec ≈ −5.1351**, and it should also be the one with the huge break. The neighbour galaxy should give a much smaller number.

**Numbers to compare with (from the paper):**
- red window: 140 ± 5 nJy
- blue window: 18 ± 4 nJy
- break strength: 7.7 (+2.3 / −1.4)

In [ ]:
files = ["spec_150135.fits", "spec_948311.fits"]
results = {}

for name in files:
    wave, flux, err, ra, dec = load_spectrum(name)
    r = balmer_break(wave, flux, err)
    results[name] = (wave, flux, err, r)
    print(name)
    print(f"   position: RA = {ra}, Dec = {dec}")
    print(f"   red window  (4000-4100 Å): {r['red']*1000:6.1f} ± {r['red_err']*1000:4.1f} nJy  ({r['n_red']} pixels)")
    print(f"   blue window (3620-3720 Å): {r['blue']*1000:6.1f} ± {r['blue_err']*1000:4.1f} nJy  ({r['n_blue']} pixels)")
    print(f"   Balmer break strength = {r['value']:.1f}  (+{r['plus']:.1f} / -{r['minus']:.1f})")
    print()

bh_file = max(results, key=lambda k: results[k][3]["value"])
print("File with the huge break (should be MoM-BH*-1):", bh_file)
print("Paper: red 140 ± 5 nJy, blue 18 ± 4 nJy, break 7.7 (+2.3 / -1.4)")

## Step 4 — Plot the spectrum

The black line is the spectrum and the grey band is its error bar. The blue and red shaded strips are the two windows. The dashed lines mark the Balmer limit, Hγ and Hβ, shifted by the redshift. You should see the light stay very faint to the left of the Balmer limit and then climb steeply.

The plot is also saved as **MoM-BH1_balmer_break.png**. To keep it, click the folder icon on the left side of Colab, then the ⋮ next to the file, then **Download**.

In [ ]:
wave, flux, err, r = results[bh_file]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.step(wave, flux * 1000, where="mid", color="black", lw=1, label="MoM-BH*-1 (JWST prism)")
ax.fill_between(wave, (flux - err) * 1000, (flux + err) * 1000, step="mid", color="gray", alpha=0.3)

# shade the two windows
for window, color, label in [(BLUE_WINDOW, "tab:blue", "blue window"), (RED_WINDOW, "tab:red", "red window")]:
    lo, hi = [w * (1 + Z) / 1e4 for w in window]
    ax.axvspan(lo, hi, color=color, alpha=0.3, label=f"{label} ({window[0]}-{window[1]} Å rest)")

# set the vertical range from the data between 1 and 5.3 µm
show = (wave > 1.0) & (wave < 5.3)
top = 1.15 * np.max(flux[show] * 1000)
ax.set_ylim(-0.1 * top, top)

# mark hydrogen features, shifted to observed wavelengths
for line, label in [(3647, "Balmer limit"), (4342, "Hγ"), (4863, "Hβ")]:
    x = line * (1 + Z) / 1e4
    ax.axvline(x, color="gray", ls="--", lw=0.8)
    ax.text(x, 0.97 * top, label, rotation=90, ha="right", va="top", fontsize=9)

ax.set_xlim(1.0, 5.3)
ax.set_xlabel("Observed wavelength (µm)")
ax.set_ylabel("Brightness (nJy)")
ax.set_title(f"MoM-BH*-1: Balmer break = {r['value']:.1f} (+{r['plus']:.1f} / -{r['minus']:.1f})")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig("MoM-BH1_balmer_break.png", dpi=150)
plt.show()

## What your result means

- **About 7–8:** you've reproduced the paper. Your pipeline works, and you can trust it for the next steps (temperature, luminosity, size).
- **Somewhat different, but inside the error bars:** normal. Small choices, like how pixels at the window edges are counted, shift the number a little.
- **Very different (for example, 2 or 20):** check the redshift, the window numbers, and that you're looking at the right file. If you get stuck, copy the printed output or error message and ask for help.

If the nJy numbers look 1000× too big or too small, the file uses different flux units. The break strength is a ratio, so it isn't affected.